# Week 11 Lab (Instructor Copy)
This guide mirrors the student notebook with facilitation notes and expected observations. Encourage students to verbalise what they see after each cell.

## Facilitation Reminders
- Model the first cell run for each section.
- Prompt students to explain outputs before you reveal interpretations.
- Celebrate observations more than code — today is about understanding concepts, not writing new functions.

## Key Vocabulary (Review Aloud)
- File extension, metadata, sequential access, direct access, disk scheduling.

### Library Cheat Sheet (review with students)
- `Counter`: behaves like a tally dictionary. Ask what else we could count.
- `Path`: demo `.suffix`, `.name`, `.parent`.
- `random`: explain seeding for reproducible demos.
- `struct`: show how bytes turn into numbers.
- `time`: use `perf_counter()` terminology.

In [ ]:
# ⚙️ Setup — just run this cell once. No edits needed!
from collections import Counter
from pathlib import Path
import random
import struct
import time

FILES = [
    'budget_2024.xlsx', 'meeting_notes.txt', 'course_outline.pdf', 'diagram.vsdx',
    'photo.jpg', 'archive.tar.gz', 'script.sh', 'app.exe', 'data_dump.csv',
    'design_mockup.psd', 'music_track.flac', 'index.html', 'style.css', 'report_final.docx',
    'presentation.pptx', 'readme.md', 'analysis.ipynb', 'thumbnail.png', 'backup.bak',
    'invoice_01.pdf', 'logs/system.log', 'malware_sample.pdf.exe', 'config.yaml',
]

REQUESTS = [98, 183, 37, 122, 14, 124, 65, 67]
START_HEAD = 53

lab_dir = Path('week_11_lab_data')
lab_dir.mkdir(exist_ok=True)
sample_path = lab_dir / 'sequential_random.bin'
if not sample_path.exists():
    random.seed(42)
    with sample_path.open('wb') as fh:
        for _ in range(50_000):
            fh.write(struct.pack('<I', random.randint(0, 1_000_000)))
print('Setup complete. File created at', sample_path)


## Part 1 · Extension Detective (Instructor Notes)
- Before running, ask students to predict which categories will dominate.
- After running, cold-call for definitions (e.g., “What is a `.csv` used for?”).

In [ ]:
def show_extension_summary(file_names):
    """Print a friendly summary of file categories and return details."""
    categories = {
        '.txt': 'document', '.pdf': 'document', '.docx': 'document', '.pptx': 'presentation',
        '.xlsx': 'spreadsheet', '.csv': 'data', '.json': 'data', '.yaml': 'config',
        '.md': 'documentation', '.ipynb': 'notebook', '.html': 'web', '.css': 'web',
        '.jpg': 'image', '.png': 'image', '.gif': 'image', '.psd': 'design', '.flac': 'audio',
        '.exe': 'executable', '.app': 'executable', '.sh': 'script', '.tar.gz': 'archive',
        '.bak': 'backup', '.vsdx': 'diagram', '.log': 'log'
    }

    totals = Counter()
    details = []
    for name in file_names:
        suffix = '.tar.gz' if name.endswith('.tar.gz') else Path(name).suffix
        category = categories.get(suffix, 'unknown')
        totals[category] += 1
        details.append({'filename': name, 'extension': suffix or '(none)', 'category': category})

    print('Category counts:')
    for cat, count in totals.most_common():
        print(f"  {cat:<12} — {count}")
    return details

extension_details = show_extension_summary(FILES)
extension_details[:8]


Expected takeaway: documents, images, and data files dominate. Reinforce that `unknown` results require manual inspection.

### Reflection Prompt
Invite students to share habits for double-checking suspicious categories (preview, VirusTotal, sandbox).

### Suspicious Filename Discussion
Run the scan and narrate each rule. Ask, “Why is a double extension dangerous?”

In [ ]:
def flag_suspicious(file_names):
    """Return a list of filenames that deserve a closer look."""
    red_flags = []
    for name in file_names:
        check = name.lower()
        pieces = name.split('.')
        if check.endswith('.exe') and len(pieces) > 2:
            red_flags.append((name, 'Double extension ending in .exe'))
        elif len(pieces) > 2 and not check.endswith('.tar.gz'):
            red_flags.append((name, 'Multiple dots — verify true type'))
    return red_flags

suspects = flag_suspicious(FILES)
suspects


Talking point: emphasise safe verification (upload to scanning service, inspect hashes) and policy-driven deletion.

## Part 2 · Sequential vs. Direct Access
- Encourage predictions before running.
- If random access is faster (common on SSDs), use it to differentiate HDD vs. SSD behaviour.

In [ ]:
def read_sequential(path):
    start = time.perf_counter()
    total = 0
    with path.open('rb') as fh:
        chunk = fh.read(4)
        while chunk:
            total += struct.unpack('<I', chunk)[0]
            chunk = fh.read(4)
    elapsed = time.perf_counter() - start
    return total, elapsed

seq_total, seq_time = read_sequential(sample_path)
print(f'Sequential read finished in {seq_time:.4f} seconds')


In [ ]:
def read_random(path, samples=500):
    size = path.stat().st_size
    start = time.perf_counter()
    total = 0
    with path.open('rb') as fh:
        for _ in range(samples):
            offset = random.randint(0, (size // 4) - 1) * 4
            fh.seek(offset)
            data = fh.read(4)
            total += struct.unpack('<I', data)[0]
    elapsed = time.perf_counter() - start
    return total, elapsed

rand_total, rand_time = read_random(sample_path)
print(f'Random access finished in {rand_time:.4f} seconds')


In [ ]:
print(f'Sequential time: {seq_time:.4f} seconds')
print(f'Random time:      {rand_time:.4f} seconds')
print('Tip: note these numbers in your writing log for later.')
print('Instructor note: compare SSD vs. HDD experiences shared by students.')


Debrief: solicit explanations referencing caching, seek time, and device type.

## Part 3 · Disk Scheduling Snapshot
- Run once, then redraw the head movements on the board.
- Ask different groups to explain each algorithm in plain language.

In [ ]:
def fcfs(start, requests):
    order = list(requests)
    movement = 0
    current = start
    waits = []
    for idx, track in enumerate(order):
        movement += abs(track - current)
        waits.append(movement if idx > 0 else abs(track - current))
        current = track
    avg_wait = sum(waits) / len(waits)
    return order, movement, avg_wait


def sstf(start, requests):
    remaining = list(requests)
    order = []
    current = start
    movement = 0
    wait_accum = 0
    while remaining:
        next_track = min(remaining, key=lambda track: abs(track - current))
        remaining.remove(next_track)
        movement += abs(next_track - current)
        wait_accum += movement
        current = next_track
        order.append(next_track)
    avg_wait = wait_accum / len(order)
    return order, movement, avg_wait


def scan(start, requests, max_track=199):
    up = sorted([r for r in requests if r >= start])
    down = sorted([r for r in requests if r < start], reverse=True)

    order = []
    movement = 0
    current = start
    wait_accum = 0

    for track in up:
        movement += abs(track - current)
        wait_accum += movement
        current = track
        order.append(track)

    if down:
        movement += abs(max_track - current)
        current = max_track
        for track in down:
            movement += abs(track - current)
            wait_accum += movement
            current = track
            order.append(track)

    avg_wait = wait_accum / len(order) if order else 0
    return order, movement, avg_wait

fcfs_stats = fcfs(START_HEAD, REQUESTS)
sstf_stats = sstf(START_HEAD, REQUESTS)
scan_stats = scan(START_HEAD, REQUESTS)

results = {
    'FCFS': fcfs_stats,
    'SSTF': sstf_stats,
    'SCAN': scan_stats,
}

for name, (order, movement, wait) in results.items():
    print(f"{name:<5} | order {order} | movement {movement} | avg wait {wait:.2f}")


In [ ]:
print('Summary table for your notes:')
print(f"FCFS -> movement {fcfs_stats[1]} | avg wait {fcfs_stats[2]:.2f}")
print(f"SSTF -> movement {sstf_stats[1]} | avg wait {sstf_stats[2]:.2f}")
print(f"SCAN -> movement {scan_stats[1]} | avg wait {scan_stats[2]:.2f}")
print('Instructor note: highlight fairness vs. efficiency trade-offs.')


Wrap-up: connect metrics to real-world scenarios (personal laptop vs. database server). Encourage students to note insights for writing log.

## Exit Reminders
- Ensure students record their reflections.
- Preview the independent home lab and writing log expectations.